# DSP - Audio Segmentation

## Import

In [1]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')

In [3]:
import glob
import os
import random

import librosa
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as signal
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm


## Configuration

### Random Seed Config

In [5]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

### File Dir Config

In [6]:
current_path = os.getcwd()
data_dir = f"{current_path}/dataset/VCTK-Corpus/wav48"

### Model Config

In [16]:
MODEL_NAME = "dymn10_as"

### Training Config

In [7]:

EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-4

### Weights and Biases (wandb) Config

In [26]:
# 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

In [27]:
import wandb

wandb.login(relogin=True)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\trung\_netrc


True

## Load model

In [18]:
from models.dymn.model import get_model as get_dymn
model = get_dymn(pretrained_name=MODEL_NAME)
model.classifier = model.classifier[:4]

d:\Projects\DSP\DSP\.venv\Lib\site-packages\torchvision\ops\misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(


In [9]:
%%capture

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [10]:
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

Model's state_dict:
layers.0.depth_conv.weight 	 torch.Size([1, 1, 4, 144])
layers.0.depth_conv.residuals.0.weight 	 torch.Size([4, 32])
layers.0.depth_conv.residuals.0.bias 	 torch.Size([4])
layers.0.depth_norm.weight 	 torch.Size([16])
layers.0.depth_norm.bias 	 torch.Size([16])
layers.0.depth_norm.running_mean 	 torch.Size([16])
layers.0.depth_norm.running_var 	 torch.Size([16])
layers.0.depth_norm.num_batches_tracked 	 torch.Size([])
layers.0.depth_act.lambdas 	 torch.Size([4])
layers.0.depth_act.init_v 	 torch.Size([4])
layers.0.depth_act.coef_net.0.weight 	 torch.Size([64, 32])
layers.0.depth_act.coef_net.0.bias 	 torch.Size([64])
layers.0.proj_conv.weight 	 torch.Size([1, 1, 4, 256])
layers.0.proj_conv.residuals.0.weight 	 torch.Size([4, 32])
layers.0.proj_conv.residuals.0.bias 	 torch.Size([4])
layers.0.proj_norm.weight 	 torch.Size([16])
layers.0.proj_norm.bias 	 torch.Size([16])
layers.0.proj_norm.running_mean 	 torch.Size([16])
layers.0.proj_norm.running_var 	 torch.Size([16

In [19]:
dummy_input = torch.randn(1, 1, 128, 1024)

try:
    output = model(dummy_input)
    print(f"Input shape {dummy_input.shape} valid! Output shape: {output[0].shape}")
    del output
except RuntimeError as e:
    print("Shape mismatch error:", e)

Input shape torch.Size([1, 1, 128, 1024]) valid! Output shape: torch.Size([1, 1280])


## Load Dataset

In [12]:
def get_speaker_splits(root_dir, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    Partition dataset by Speaker IDs to guarantee open-set evaluation.
    """
    # 1. Gather all unique speaker IDs
    speaker_dirs = sorted(glob.glob(os.path.join(root_dir, 'p*')))
    speakers = [os.path.basename(d) for d in speaker_dirs]
    
    # 2. Shuffle deterministically
    rng = random.Random(seed)
    rng.shuffle(speakers)
    
    # 3. Compute 8:1:1 split index boundaries
    total_speakers = len(speakers)
    train_end = int(total_speakers * train_ratio)
    val_end = train_end + int(total_speakers * val_ratio)
    
    train_speakers = set(speakers[:train_end])
    val_speakers = set(speakers[train_end:val_end])
    test_speakers = set(speakers[val_end:])
    
    return train_speakers, val_speakers, test_speakers

In [13]:
class AudioPreprocessor(nn.Module):
    def __init__(self, sample_rate=16000, n_mels=128, target_frames=1024):
        super().__init__()
        self.target_frames = target_frames
        
        # 1. Mel Spectrogram Extractor (Frequency height = 128)
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=1024,
            win_length=512,
            hop_length=160,
            n_mels=n_mels
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def forward(self, waveform):
        """
        Input waveform shape: (batch_size, num_samples) or (num_samples,)
        Output tensor shape:   (batch_size, 1, 128, 1024)
        """
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)  # Shape: (1, num_samples)

        # Generate Log-Mel Spectrogram -> Shape: (batch_size, 128, T)
        mel = self.mel_spectrogram(waveform)
        log_mel = self.amplitude_to_db(mel)

        # Add Channel Dimension -> Shape: (batch_size, 1, 128, T)
        x = log_mel.unsqueeze(1)

        # Rescale Time Dimension T to target 1024 frames -> Shape: (batch_size, 1, 128, 1024)
        if x.shape[-1] != self.target_frames:
            x = F.interpolate(
                x, 
                size=(128, self.target_frames), 
                mode='bilinear', 
                align_corners=False
            )

        # Z-score Normalization per sample
        mean = x.mean(dim=(-2, -1), keepdim=True)
        std = x.std(dim=(-2, -1), keepdim=True) + 1e-6
        x = (x - mean) / std

        return x

In [14]:
class VCTKTripletDataset(Dataset):
    def __init__(self, root_dir, allowed_speakers, sample_rate=16000, segment_len_sec=0.6, 
                 transform=None, subset_ratio=0.05, seed=42):
        self.root_dir = root_dir
        self.sample_rate = sample_rate
        self.segment_samples = int(sample_rate * segment_len_sec)
        self.transform = transform

        # Standardize random selection across runs
        rng = random.Random(seed)

        self.speaker_to_files = {}
        speaker_dirs = sorted(glob.glob(os.path.join(root_dir, 'p*')))
        
        for s_dir in speaker_dirs:
            speaker_id = os.path.basename(s_dir)
            if speaker_id not in allowed_speakers:
                continue
            files = sorted(glob.glob(os.path.join(s_dir, '*.flac')) + glob.glob(os.path.join(s_dir, '*.wav')))
            
            if len(files) >= 2:
                # Calculate 5% count, guaranteeing at least 2 files for Anchor/Positive pairs
                target_count = max(2, int(len(files) * subset_ratio))
                selected_files = rng.sample(files, target_count)
                
                self.speaker_to_files[speaker_id] = selected_files
        
        self.speakers = list(self.speaker_to_files.keys())
    def _load_and_crop(self, file_path):
        waveform, sr = torchaudio.load(file_path)
        
        # Resample if needed
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        # Convert to mono if multi-channel
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Random Crop or Zero-Pad to target segment length
        num_samples = waveform.shape[-1]
        if num_samples >= self.segment_samples:
            max_start = num_samples - self.segment_samples
            start = random.randint(0, max_start)
            cropped = waveform[:, start:start + self.segment_samples]
        else:
            padding = self.segment_samples - num_samples
            cropped = torch.nn.functional.pad(waveform, (0, padding))

        return cropped.squeeze(0)  # Shape: (segment_samples,)

    def __len__(self):
        # Defines epoch length based on total estimated audio clips
        return sum(len(files) for files in self.speaker_to_files.values())

    def __getitem__(self, idx):
        # Select Anchor Speaker (A)
        anchor_speaker = random.choice(self.speakers)
        
        # Pick 2 different clips for Positive pair (or same clip cropped at different positions)
        a_file, p_file = random.sample(self.speaker_to_files[anchor_speaker], 2)

        # Select Negative Speaker (N != A)
        negative_speaker = random.choice([s for s in self.speakers if s != anchor_speaker])
        n_file = random.choice(self.speaker_to_files[negative_speaker])

        # Load cropped raw audio
        a_wave = self._load_and_crop(a_file)
        p_wave = self._load_and_crop(p_file)
        n_wave = self._load_and_crop(n_file)

        # Apply DSP pipeline preprocessing if specified
        if self.transform:
            a_wave = self.transform(a_wave.numpy())
            p_wave = self.transform(p_wave.numpy())
            n_wave = self.transform(n_wave.numpy())

        return a_wave, p_wave, n_wave

In [15]:
# Get speaker lists
train_spk, val_spk, test_spk = get_speaker_splits(data_dir, train_ratio=0.8, val_ratio=0.1, seed=SEED)

# Build Datasets
train_ds = VCTKTripletDataset(data_dir, allowed_speakers=train_spk, subset_ratio=0.05)
val_ds   = VCTKTripletDataset(data_dir, allowed_speakers=val_spk, subset_ratio=0.05)
test_ds  = VCTKTripletDataset(data_dir, allowed_speakers=test_spk, subset_ratio=0.05)

# Create Loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
def extract_embedding(model, x):
    output = model(x)
    # If DyMN returns (logits, features), select the feature tensor (e.g., shape [B, 1280 or 960])
    features = output[1] if isinstance(output, tuple) else output

    # L2-normalize vectors for stable distance calculation
    return F.normalize(features, p=2, dim=1)

## Train

In [ ]:
criterion = nn.TripletMarginLoss(margin=0.3, p=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [ ]:
run = wandb.init(
    project="DSP", 
    entity="hieu040390-fpt-university", 
    name="dymn10_as_triplet_training",
    config={
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
    })

In [ ]:
model.train()
for epoch in tqdm(range(EPOCHS), total=EPOCHS, desc="Training Progress"):
    running_loss = 0.0
    for batch_idx, (anchor, positive, negative) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}"):
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()

        # Preprocess raw audio to log-mel spectrograms
        preprocessor = AudioPreprocessor()
        anchor_spec = preprocessor(anchor)
        positive_spec = preprocessor(positive)
        negative_spec = preprocessor(negative)

        # Forward pass through the model to get embeddings
        anchor_emb = extract_embedding(model, anchor_spec)
        positive_emb = extract_embedding(model, positive_spec)
        negative_emb = extract_embedding(model, negative_spec)

        # Compute triplet loss
        loss = criterion(anchor_emb, positive_emb, negative_emb)

        # Backpropagation and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    for batch_idx, (anchor, positive, negative) in tqdm(enumerate(val_loader), total=len(val_loader), desc=f"Validation Epoch {epoch+1}/{EPOCHS}"):
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        # Preprocess raw audio to log-mel spectrograms
        preprocessor = AudioPreprocessor()
        anchor_spec = preprocessor(anchor)
        positive_spec = preprocessor(positive)
        negative_spec = preprocessor(negative)

        # Forward pass through the model to get embeddings
        anchor_emb = extract_embedding(model, anchor_spec)
        positive_emb = extract_embedding(model, positive_spec)
        negative_emb = extract_embedding(model, negative_spec)

        # Compute triplet loss
        val_loss = criterion(anchor_emb, positive_emb, negative_emb)

In [ ]:
with model.zero_grad():
    